# Spontaneous Raman Spectra Pipeline Template

below listed out all the helper functions wrote

1. from helpers import save_and_load
    1. save_spectrum(fileType: 'txt' or 'csv', outputpath as Path variable, wavenumber as nparray, intensity as nparray, [optional] background curve as nparray, header defaults to False means no header) -> None:
        saves the wavenumber and intensity data to a file either csv or txt format
    2. read_spectrum(path as Path variable, header defaults to False) -> Tuple[np.ndarry]
        read a spectrum file, the first coln will be wavenumber, second coln will be intensity, [optional] third coln will be background
    3. save_ratios(path as Path variable, data as a List of Dict[str, any], header defaults to true) -> None:
        saves a list of dictionaries containing ratio data to a csv file
    4. load_ratios(path as Path variable) -> pd.DataFrame:
        loads ratio data from a csv or a txt file into a pandas DataFrame
2. from helpers import preprocessing
    1. fitted_bg_subtraction(wavenumber as nparray, intensity as nparray, range start defaults to 1000, range end defaults to 3900) -> Tuple[nparray, nparray, nparry]:
        performs a background subtraction by fitting a Gaussian curve to the wavenumber range from (range_start to range_end), saves the fitted Gaussia curve to the third nparray. This function only fits to one Gaussian peak
    2. spectra_bg_subtraction(wavenumber as nparray, intensity as nparray, background spectra wavenumber as nparray, background spectra intensity sa nparray) -> Tuple[nparray, nparray]:
        performs a background subtraction if you provide a background spectra, no background fitting
    3. simple_baseline_correction(wavenumber as nparray, intensity as nparray, cell_silent_start as int, cell_silent_end as int, trim defaults to None, min_max defaults to True) -> Tuple[nparray, nparray]:
        performs a simple baseline correction on a Raman spectrum by setting the average intensity value from 2300 to 2600 as zero, everything below will be set to zero. min_max True if perform a min-max normalization
    4. polynomial_baseline_correction(wavenumber as nparray, intensity as nparray) -> Tuple[nparray, nparray]:
        performs a baseline correction by fitting a polynomial to the spectrum
    5. smoothing(wavenumber as nparray, intensity as nparray, window_length as int defaults to 11, polyorder as int defaults to 2) -> Tuple[nparray, nparray]:
        performs a savitzky-golay filter for a spectrum to smooth
3. from helpers import analysis
    1. peak_ratio_calculator(wavenumber as nparray, intensity as nparray, peak 1 location, peak 2 location, [optional] background intensity) -> float:
        calculate a ratio between the intensity at two peak wavenumber location. If a background intensity is provided, peak 2 will be seen as a water background
    2. area_under_curve_ratio_calculator(wavenumber as nparry, intensity as nparray, peak range 1, peak range 2, [optional] background intensity) -> float:
        calculate the area underr the curve for a specified range of wavenumber in a Raman spectrum. If a background intensity is provided, peak 2's intensity will be read from background intensity
    3. t_test_calculator(group_1 as nparray, group_2 as nparray) -> tuple:
        performs a t-test between two groups of data
4. from helpers import plots
    
    nparray is able to used by matplotlib, for all this part, please see the script yourself since might be a bit complicate to explain here.

### Library Loading

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd

os.chdir(os.getcwd())

from helpers import save_and_load as sl
from helpers import preprocessing as prep
from helpers import analysis
from helpers import plots
import pathlib as Path

Load file path to a df table

In [ ]:
base_dir = r'D:\Chrome\Workspace\Projects\Rapamycin\data\spontaneous\Rapamycin_2'
base_dir = Path.Path(base_dir)
all_files = sorted(base_dir.glob("*/*/*/*.txt"))
rows = []

for path in all_files:
    rel = path.relative_to(base_dir)
    day, organ, group, filename = rel.parts
    if organ != 'Gut':
        continue

    name_lower = path.name.lower()
    if ("fitted" in name_lower or "baseline" in name_lower or "bg" in name_lower or "background" in name_lower):
        continue
    
    if ("ctrl" not in name_lower and "rapa" not in name_lower):
        continue

    bg_path = path.with_name(path.stem + '_BG.txt')
    if not bg_path.exists():
        print(f"Warning: Background file {bg_path} does not exist for {path}")
        continue
    
    rows.append({
        'day': day.split('_')[1],
        'organ': organ,
        'group': group,
        'filename': path.stem,
        'spectrum_path': path,
        'background_path': bg_path,
        'has_background': bg_path.exists()
    })

file_table = pd.DataFrame(rows)
gut_file_table = file_table.copy().reset_index(drop=True)

with open('file_table.csv', 'w') as f:
    gut_file_table.to_csv(f, index=False)

gut_file_table.head(20)

## Gut Analysis Plan

1. Keep only gut spectra for the active analysis.
2. For each day/group, merge spectra by summing raw intensities into one total spectrum.
3. Run fitted background subtraction on each merged spectrum.
4. Min-max normalize, then apply Savitzky-Golay smoothing.
5. Plot the requested comparisons: `1_RAPA` vs `4_CTRL`, and `2_RAPA` vs `5_CTRL`.

### Gut Analysis: merge, preprocess, and plot

In [ ]:
# Gut-only merged spectra workflow
# Change this list if you only want one day, e.g. analysis_days = ["D10"]
analysis_days = sorted(gut_file_table['day'].unique())
gut_groups = ["1_RAPA", "2_RAPA", "4_CTRL", "5_CTRL"]
comparison_pairs = [("1_RAPA", "4_CTRL"), ("2_RAPA", "5_CTRL")]

# Savitzky-Golay smoothing settings
savgol_window = 21
savgol_polyorder = 3

# Plot ranges
whole_spectrum_region = (1000, 3900)
zoom_regions = {
    "Fingerprint": (1000, 1800),
    "Cell silent": (2100, 2300),
    "CH stretching": (2800, 3050),
}

group_colors = {
    "1_RAPA": "#0072B2",
    "2_RAPA": "#009E73",
    "4_CTRL": "#D55E00",
    "5_CTRL": "#CC79A7",
}


def merge_group_spectra(group_files):
    # Sum all raw spectra in one group onto a shared wavenumber axis.
    if group_files.empty:
        raise ValueError("No spectra found for this group.")

    reference_wn = None
    merged_intensities = []

    for spectrum_path in group_files['spectrum_path']:
        wn, intensity = sl.read_spectrum(Path.Path(spectrum_path))
        wn = np.asarray(wn, dtype=float)
        intensity = np.asarray(intensity, dtype=float)

        valid = np.isfinite(wn) & np.isfinite(intensity)
        wn = wn[valid]
        intensity = intensity[valid]

        order = np.argsort(wn)
        wn = wn[order]
        intensity = intensity[order]

        if reference_wn is None:
            reference_wn = wn
            intensity_on_reference = intensity
        elif len(wn) == len(reference_wn) and np.allclose(wn, reference_wn):
            intensity_on_reference = intensity
        else:
            intensity_on_reference = np.interp(reference_wn, wn, intensity)

        merged_intensities.append(intensity_on_reference)

    total_intensity = np.sum(np.vstack(merged_intensities), axis=0)
    return reference_wn, total_intensity, len(merged_intensities)


def min_max_normalize(intensity):
    intensity = np.asarray(intensity, dtype=float)
    intensity_min = np.nanmin(intensity)
    intensity_max = np.nanmax(intensity)
    if np.isclose(intensity_max, intensity_min):
        return np.zeros_like(intensity)
    return (intensity - intensity_min) / (intensity_max - intensity_min)


def preprocess_merged_spectrum(wn, total_intensity):
    fit_wn, bg_subtracted, bg_curve = prep.fitted_bg_subtraction(
        wn,
        total_intensity,
        range_start=whole_spectrum_region[0],
        range_end=whole_spectrum_region[1],
    )
    if fit_wn.size == 0:
        raise ValueError("Background fitting returned an empty spectrum.")

    normalized = min_max_normalize(bg_subtracted)
    smooth_wn, smoothed = prep.smoothing(
        fit_wn,
        normalized,
        window_length=savgol_window,
        polyorder=savgol_polyorder,
    )

    return {
        "wn": smooth_wn,
        "intensity": smoothed,
        "normalized_intensity": normalized,
        "bg_subtracted_intensity": bg_subtracted,
        "background_curve": bg_curve,
    }


def plot_gut_pair(day, group_a, group_b):
    fig = plt.figure(figsize=(14, 8), constrained_layout=True)
    gs = fig.add_gridspec(2, 3, height_ratios=[1.25, 1])
    full_ax = fig.add_subplot(gs[0, :])
    zoom_axes = [fig.add_subplot(gs[1, idx]) for idx in range(3)]

    for group in (group_a, group_b):
        spectrum = gut_processed_spectra[(day, group)]
        full_ax.plot(
            spectrum["wn"],
            spectrum["intensity"],
            label=group,
            color=group_colors[group],
            linewidth=1.8,
        )

    full_ax.set_title(f"Gut {day}: {group_a} vs {group_b} - whole spectrum")
    full_ax.set_xlim(*whole_spectrum_region)
    full_ax.set_xlabel("Raman Shift (cm$^{-1}$)")
    full_ax.set_ylabel("Normalized intensity (a.u.)")
    full_ax.grid(alpha=0.25)
    full_ax.legend(frameon=False)

    for ax, (region_name, xlim) in zip(zoom_axes, zoom_regions.items()):
        for group in (group_a, group_b):
            spectrum = gut_processed_spectra[(day, group)]
            ax.plot(
                spectrum["wn"],
                spectrum["intensity"],
                label=group,
                color=group_colors[group],
                linewidth=1.6,
            )
        ax.set_title(region_name)
        ax.set_xlim(*xlim)
        ax.set_xlabel("Raman Shift (cm$^{-1}$)")
        ax.set_ylabel("Normalized intensity (a.u.)")
        ax.grid(alpha=0.25)

    return fig


gut_merged_raw = {}
gut_processed_spectra = {}
summary_rows = []

for day in analysis_days:
    day_table = gut_file_table[gut_file_table['day'].eq(day)]

    for group in gut_groups:
        group_files = day_table[day_table['group'].eq(group)]
        if group_files.empty:
            print(f"Skipping Gut {day} {group}: no files found")
            continue

        wn, total_intensity, n_spectra = merge_group_spectra(group_files)
        gut_merged_raw[(day, group)] = {
            "wn": wn,
            "total_intensity": total_intensity,
            "n_spectra": n_spectra,
        }
        gut_processed_spectra[(day, group)] = preprocess_merged_spectrum(wn, total_intensity)
        summary_rows.append({"day": day, "group": group, "merged_spectra": n_spectra})

gut_summary = pd.DataFrame(summary_rows)
try:
    display(gut_summary)
except NameError:
    print(gut_summary)

for day in analysis_days:
    for group_a, group_b in comparison_pairs:
        if (day, group_a) in gut_processed_spectra and (day, group_b) in gut_processed_spectra:
            plot_gut_pair(day, group_a, group_b)
            plt.show()
        else:
            print(f"Skipping Gut {day} {group_a} vs {group_b}: one or both spectra are missing")